In [2]:
import pandas as pd

# 1. 파일 로드
df = pd.read_csv('jeju_places_verified.csv', encoding='cp949')
original_count = len(df)

# 2. URL 중복 제거 (11건)
df = df.drop_duplicates(subset=['place_url'], keep='first')

# 3. 숙박 시설 및 여행 부적합 키워드 제거
# 이름(name) 컬럼에서 아래 단어가 포함된 행을 모두 삭제합니다.
remove_keywords = [
    '호텔', '펜션', '리조트', '게스트하우스', '민박', '스테이', '콘도', '모텔', 
    '캠핑', '글램핑', '빌라', '아파트', '오피스텔', '원룸'
]
# 관공서 및 기타 노이즈 제거
noise_keywords = ['동사무소', '주민센터', '보건소', '학교', '학원', '우체국']

all_exclude = remove_keywords + noise_keywords
df = df[~df['name'].str.contains('|'.join(all_exclude), na=False)]

# 4. 올레길 등 코스형 데이터 제거
df = df[~df['name'].str.contains('올레|코스', na=False)]

print(f"원본 데이터: {original_count}개")
print(f"정제 후 데이터: {len(df)}개 (약 {original_count - len(df)}개 삭제됨)")

# 5. 정제된 파일 저장
df.to_csv('jeju_places_cleaned.csv', index=False, encoding='utf-8-sig')

원본 데이터: 1394개
정제 후 데이터: 1280개 (약 114개 삭제됨)


In [3]:
import pandas as pd

# 1. 데이터 불러오기
df = pd.read_csv('jeju_places_cleaned.csv')

# 2. 제거할 키워드 집합 (쇼핑, 의료, 금융, 생활인프라 등)
last_noise_keywords = [
    '나이키', '아디다스', '하이마트', '디지털프라자', '베스트샵', '다이소',
    '약국', '병원', '의원', '치과', '내과', '한의원', '은행', '농협', '신협',
    '우체국', '주차장', '충전소', '마을회관', '아파트', '빌라', '부동산',
    '서비스센터', '정비', '카센타', '공업사', '안경원', '마트', '슈퍼'
]

# 3. 키워드 필터링 적용
df_final = df[~df['name'].str.contains('|'.join(last_noise_keywords), na=False)]

# 4. 종교시설 처리 (유명한 곳은 제외하고 일반 교회/성당 제거)
# 방주교회, 순례자의 교회, 약천사, 관음사는 남깁니다.
famous_religious = ['방주교회', '순례자의 교회', '약천사', '관음사']
df_final = df_final[
    (~df_final['name'].str.contains('교회|성당|사찰|절', na=False)) | 
    (df_final['name'].str.contains('|'.join(famous_religious), na=False))
]

# 5. 최종 결과 확인 및 저장
print(f"최종 정제 완료: {len(df_final)}개 데이터")
df_final.to_csv('jeju_places_final_fixed.csv', index=False, encoding='utf-8-sig')

최종 정제 완료: 1269개 데이터


In [2]:
import pandas as pd

# 파일 로드
df = pd.read_csv('jeju_places_final_fixed.csv',encoding='cp949')

# 1. 제거할 키워드 (브랜드, 숙박, 코스형)
final_noise = [
    '캉골', '푸마', 'NBA', 'PXG', '펠리스타운', '파크텔', '휘닉스', 
    '둘레길', '인증센터', '둘레길', '올레길'
]

# 2. 지리적 제거 (추자도, 마라도)
island_noise = ['추자도', '마라도']

# 3. 통합 필터링 적용
df_final = df[~df['name'].str.contains('|'.join(final_noise + island_noise), na=False)]

# 4. 좌표 기반 필터링 (추자도 등 위도 33.6 이상 완전 제거)
df_final = df_final[(df_final['lat'] < 33.6) & (df_final['lat'] > 33.0)]

print(f"최종 정제 완료: {len(df_final)}개")
df_final.to_csv('jeju_places_scraping_ready_v4.csv', index=False, encoding='utf-8-sig')

최종 정제 완료: 1250개


In [4]:
import pandas as pd

df = pd.read_csv('jeju_places_scraping_ready_v4.csv')

# 진짜 마지막으로 긁어낸 노이즈 키워드들
the_last_noise = [
    '호스텔', '백패커스', '빌리지', '수련관', '야영장', # 숙박/공공
    '선착장', '성산항', '항구', # 이동 거점
    '메트로시티', '톰보이', '대리점' # 브랜드/쇼핑
]

# 필터링 적용
df_final = df[~df['name'].str.contains('|'.join(the_last_noise), na=False)]

# 좌표 최종 확인 (이미 깨끗함)
df_final = df_final[(df_final['lat'] < 33.6) & (df_final['lat'] > 33.1)]

print(f"최종의 최종 데이터 수: {len(df_final)}개")
df_final.to_csv('jeju_places_final_for_scraping.csv', index=False, encoding='utf-8-sig')

최종의 최종 데이터 수: 1234개
